# LLM Sampling Parameters — with REAL Claude API calls

* Temperature, Top-k, and Top-p — see them change a live model's output

---

The companion notebook `01-randomness-creativity.ipynb` shows the *math* of sampling on a
hand-built probability distribution. This notebook does the same three experiments, but against
the **real Claude API** so you can watch the parameters reshape an actual model's output.

| Parameter | One-line summary |
|-----------|------------------|
| **Temperature** | Makes the distribution sharper (safe) or flatter (creative) |
| **Top-k** | Only keep the k most probable tokens |
| **Top-p** | Only keep tokens until their probabilities add up to p |

Each section below:
1. Sets **one** sampling parameter
2. Sends the **same prompt** to Claude several times
3. Prints the **output** so you can see exactly what changed

> **Requirements:** `anthropic` and `python-dotenv` Python packages, and an `ANTHROPIC_API_KEY`
> in the project-root `.env` file.

---
## Setup — client, model, and a helper that calls Claude

### Why `claude-haiku-4-5`?

This notebook is *about* `temperature`, `top_k`, and `top_p`. The latest Opus models
(`claude-opus-4-8`, `claude-opus-4-7`) and `claude-fable-5` **reject** those sampling parameters
with a `400` error — they steer behaviour through prompting and the `effort` parameter instead.

So for a *sampling* demo we deliberately use **`claude-haiku-4-5`**, which still accepts all three.
It is also the fastest and cheapest model — ideal for a notebook that makes many small calls.

> One Claude-4 rule still applies: in a **single** request you may set *either* `temperature`
> *or* `top_p`, not both. Each experiment below sets only one parameter at a time, so we are fine.

In [1]:
# If the packages aren't installed yet, uncomment and run once:
# %pip install anthropic python-dotenv

import os
from dotenv import load_dotenv
import anthropic

In [2]:


# Load ANTHROPIC_API_KEY from the project-root .env (one folder up from week1/).
load_dotenv(dotenv_path=os.path.join('..', '.env'))

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not found. Add it to the project-root .env file.'
)

# The client reads ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

MODEL = 'claude-haiku-4-5'   # supports temperature / top_k / top_p (Opus 4.8 / 4.7 do not)
print(f'Client ready. Using model: {MODEL}')

Client ready. Using model: claude-haiku-4-5


### Calling Claude using Function

This function sends a prompt to Claude and returns just the reply text.

- **Guard clause**: raises an error if both `temperature` and `top_p` are set, since Claude 4+ models don't allow both at once.
- **Dynamic kwargs**: builds the API request dict, only including `system`, `temperature`, `top_k`, or `top_p` if you actually passed them in.
- **API call**: sends the request via `client.messages.create(**kwargs)`.
- **Text extraction**: the response contains a list of content blocks (text, tool-use, etc.); this pulls out just the `text` blocks, joins them, and strips whitespace — so you get a clean string back instead of the full response object.

The last line is just a smoke test — sends "pong" with a small token limit to confirm the setup works.

In [3]:
def generate(prompt, *, temperature=None, top_k=None, top_p=None,
             max_tokens=60, system=None):
    """Call Claude once and return the text reply.

    Only the sampling parameters you pass are sent to the API. On Claude 4+ models you
    cannot set both temperature and top_p in the same request, so we guard against it.
    """
    if temperature is not None and top_p is not None:
        raise ValueError('Set either temperature OR top_p in a single call, not both.')

    kwargs = {
        'model': MODEL,
        'max_tokens': max_tokens,
        'messages': [{'role': 'user', 'content': prompt}],
    }
    if system is not None:
        kwargs['system'] = system
    if temperature is not None:
        kwargs['temperature'] = temperature
    if top_k is not None:
        kwargs['top_k'] = top_k
    if top_p is not None:
        kwargs['top_p'] = top_p

    response = client.messages.create(**kwargs)
    return ''.join(b.text for b in response.content if b.type == 'text').strip()


# Quick connectivity check
print(generate('Reply with exactly: pong', max_tokens=10))

pong


---
## Part 1 — Temperature

**What it does:**  Temperature divides the log-probabilities before renormalising.

- `T = 0.0` → top token gets all the weight → **predictable / near-deterministic**
- `T = 1.0` → probabilities are used as-is → **balanced**
- higher `T`  → probabilities flatten → **creative / risky** (`claude-haiku-4-5` accepts `0.0`–`1.0`)

**Analogy:** a volume knob — turn down for safe, turn up for wild.

We send the **same creative prompt** at three temperatures and compare the replies.

In [4]:
PROMPT = (
    'Write a single, vivid opening sentence for a short story about the sea. '
    'Reply with only the sentence.'
)

for T in [0.0, 0.5, 1.0]:
    print(f'=== temperature = {T} ===')
    out = generate(PROMPT, temperature=T)
    print(out)
    print()

=== temperature = 0.0 ===
The sea that morning was the color of old bruises—purple and green and angry—and it whispered warnings that no one aboard the fishing vessel thought to heed.

=== temperature = 0.5 ===
The sea that morning wore the color of old bruises—purple and green and sickly yellow—and it whispered warnings that no one aboard the fishing vessel thought to heed.

=== temperature = 1.0 ===
The sea that morning wore the color of a bruise—deep purple bleeding into sickly green—and it whispered warnings that the old fisherman chose to ignore.



In [5]:
# Determinism check: at temperature 0 the SAME prompt tends to give the SAME answer.
print('temperature = 0.0, called twice:')
print(' 1:', generate(PROMPT, temperature=0.0))
print(' 2:', generate(PROMPT, temperature=0.0))
print()
print('temperature = 1.0, called twice:')
print(' 1:', generate(PROMPT, temperature=1.0))
print(' 2:', generate(PROMPT, temperature=1.0))

temperature = 0.0, called twice:
 1: The sea that morning was the color of old bruises—purple and green and angry—and it whispered warnings that no one aboard the fishing vessel thought to heed.
 2: The sea that morning was the color of old bruises—purple and green and angry—and it whispered warnings that no one aboard the fishing vessel thought to heed.

temperature = 1.0, called twice:
 1: The sea that morning wore the color of bruised steel, and it whispered warnings that the fisherman chose not to hear.
 2: The sea that morning was the color of hammered pewter, restless and hungry, as if it had swallowed something precious in the night and could not decide whether to spit it back or keep it forever.


### What to notice

- At **T = 0.0** the two calls are usually **identical or near-identical** — the model keeps
  picking its single most-likely continuation. (Note: even T=0 is not *guaranteed* bit-for-bit
  identical, but it is highly stable.)
- At **T = 1.0** the two calls **diverge** — different imagery, rhythm, and word choices.

### When to use which temperature

| Use case | Recommended T |
|----------|---------------|
| Maths, code, factual Q&A | 0.0 – 0.3 |
| Summarisation, translation | 0.3 – 0.7 |
| Creative writing | 0.7 – 1.0 |

---
## Part 2 — Top-k

**What it does:**  At each step, keep only the **k most probable tokens**, discard the rest,
renormalise, then sample.

**Analogy:** a shortlist — 'I will only consider the top k candidates, no matter how close or
far apart they are.'

- **k = 1** → only the single best token survives → effectively greedy / deterministic
- **larger k** → more candidates in play → more variety

We hold temperature high (to allow variety) and vary `top_k`.

In [6]:
PROMPT_K = (
    'Suggest one unexpected pizza topping. Reply with only the topping name.'
)

for k in [1, 5, 50]:
    print(f'=== top_k = {k} (temperature held at 1.0) ===')
    # Call a few times so the effect of the candidate-pool size is visible.
    for i in range(3):
        print(f'  {i + 1}:', generate(PROMPT_K, top_k=k, temperature=1.0, max_tokens=20))
    print()

=== top_k = 1 (temperature held at 1.0) ===
  1: Anchovy
  2: Anchovy
  3: Anchovy

=== top_k = 5 (temperature held at 1.0) ===
  1: Anchovy
  2: Anchovy
  3: Anchovy

=== top_k = 50 (temperature held at 1.0) ===
  1: Miso paste
  2: Anchovy
  3: Anchovies



### What to notice

- **k = 1** forces the model to take its single top token every step, so the three runs are
  the **same** (or nearly so) — the model can't reach for a rarer, more surprising topping.
- **k = 50** lets lower-probability tokens into the pool, so you see **more variety** across runs.

### The limitation of top-k
Top-k is a **fixed headcount** — it ignores how confident the model actually is:
- If one token has 95% probability, k=50 still drags in 49 near-zero candidates.
- If 100 tokens are roughly tied, k=50 unfairly cuts half of them.

That is exactly what top-p fixes.

---
## Part 3 — Top-p (Nucleus Sampling)

**What it does:**  Add up token probabilities from highest to lowest; stop once the running total
reaches **p**; keep only those tokens (the 'nucleus'); renormalise; sample.

**Analogy:** a guest list — 'keep inviting the most popular people until their combined popularity
covers p of the room.'

**Key advantage over top-k:** the pool size adapts automatically. Confident model → small nucleus.
Uncertain model → large nucleus.

We vary `top_p` (and do **not** set temperature in the same call — Claude 4+ allows only one).

In [14]:
PROMPT_P = (
    'Give one creative name for a coffee shop. Reply with only the name.'
)

for p in [0.10, 0.50, 1.00]:
    print(f'=== top_p = {p} ===')
    for i in range(3):
        print(f'  {i + 1}:', generate(PROMPT_P, top_p=p, max_tokens=20))
    print()

=== top_p = 0.1 ===
  1: Brew & Bloom
  2: Brew & Bloom
  3: Brew & Bloom

=== top_p = 0.5 ===
  1: Brew & Bloom
  2: Brew & Bloom
  3: Brew & Bloom

=== top_p = 1.0 ===
  1: The Midnight Roost
  2: Brew & Bloom
  3: The Brew & Bloom



### What to notice

- **p = 0.10** keeps only the few most-likely tokens → safe, repetitive names.
- **p = 1.00** opens the whole vocabulary → the most varied, surprising names.

The nucleus grows *as needed* — that is the key insight, and the reason top-p is usually
preferred over top-k in practice.

---
## Part 4 — Measuring variability

Sampling parameters change *how varied* the outputs are. Let's make that concrete: run the same
prompt many times at low vs high temperature and count how many **distinct** answers we get.

In [17]:
from collections import Counter

# A prompt whose answer is genuinely OPEN. "Name one colour" is too peaked —
# Claude lands on "blue" almost every time, so temperature has nothing to spread.
# A creative/subjective question has many competing high-probability answers,
# so raising the temperature visibly increases variety.
PROMPT_V = 'Invent a one-word name for a new colour. Reply with only the made-up word.'
N = 10

for T in [0.0, 1.0]:
    answers = [generate(PROMPT_V, temperature=T, max_tokens=10).lower() for _ in range(N)]
    counts = Counter(answers)
    print(f'temperature = {T}:  {len(counts)} distinct answer(s) out of {N} calls')
    for word, c in counts.most_common():
        print(f'    {word:<14} {c}x  {"#" * c}')
    print()

temperature = 0.0:  1 distinct answer(s) out of 10 calls
    vorn           10x  ##########

temperature = 1.0:  3 distinct answer(s) out of 10 calls
    vorn           8x  ########
    velm           1x  #
    zephyrine      1x  #



### What to notice

- At **T = 0.0** you typically get **1 distinct answer** repeated `N` times — the model locks on.
- At **T = 1.0** you get **several distinct answers** — the distribution is being explored.

This is the same lesson as the math notebook's sampling histogram, but produced by a live model.

---
## Part 5 — Experiment: try your own settings

Change the values below and re-run. Remember: set **either** `MY_TEMPERATURE` **or** `MY_TOP_P`
to `None` (you can't use both in one call). `MY_TOP_K` can be combined with either.

In [ ]:
# ── Change these and re-run ───────────────────────────────────────
MY_PROMPT      = 'Write a one-line motto for a space exploration company.'
MY_TEMPERATURE = 1.0     # 0.0 - 1.0, or None to disable
MY_TOP_K       = None    # e.g. 1, 5, 50, or None to disable
MY_TOP_P       = None    # e.g. 0.1, 0.5, 1.0, or None  (can't combine with temperature)
N_RUNS         = 3
# ───────────────────────────────────────────────────

print(f'prompt: {MY_PROMPT!r}')
print(f'temperature={MY_TEMPERATURE}, top_k={MY_TOP_K}, top_p={MY_TOP_P}\n')
for i in range(N_RUNS):
    out = generate(MY_PROMPT, temperature=MY_TEMPERATURE,
                   top_k=MY_TOP_K, top_p=MY_TOP_P, max_tokens=60)
    print(f'{i + 1}: {out}')

---
## Quick Reference

### Temperature
| Value | Effect | Good for |
|-------|--------|----------|
| 0.0 – 0.3 | Near-deterministic | Code, maths, facts |
| 0.4 – 0.7 | Natural, balanced | Summaries, email |
| 0.8 – 1.0 | Creative, varied | Chat, storytelling |

### Top-k
| Value | Effect |
|-------|--------|
| k = 1 | Greedy decoding — always the top token |
| k = 5–20 | Tight control — structured outputs |
| k = 40–100 | General use — variety without noise |

### Top-p
| Value | Effect |
|-------|--------|
| p = 0.1–0.5 | Very focused, small nucleus |
| p = 0.7–0.9 | Sweet spot — adaptive, coherent |
| p = 0.95–1.0 | Wide open — includes rarer tokens |

### Model note
`temperature`, `top_k`, and `top_p` work on `claude-haiku-4-5` (and other Claude 4.5 / earlier
models). They are **removed** on `claude-opus-4-8`, `claude-opus-4-7`, and `claude-fable-5`,
which return a `400` if you send them — there you steer behaviour with prompting and the
`effort` parameter instead. On any Claude 4+ model, a single request may set **either**
`temperature` **or** `top_p`, not both.